# Phase 5: VOID No-Gemini Colab Free Demo

This notebook runs Netflix VOID Pass 1 inference for one short research test. It does not call Gemini or the VOID VLM mask-reasoner. The expected input is an already prepared package containing `input_video.mp4` or `input.mp4`, `quadmask_0.mp4`, and `prompt.json`.

For Colab free, start with a very short clip and low-memory settings. If the free GPU fails, keep the exact error as the Phase 5 research result and retry with the smaller settings in the configuration cell.

## Execution Order

1. Check Colab GPU and runtime.
2. Clone Netflix `void-model`.
3. Install dependencies.
4. Download the CogVideoX base model and VOID Pass 1 checkpoint.
5. Put the Phase 5 zip in Google Drive, `/content`, or a direct URL.
6. Validate the video, quadmask, and prompt.
7. Run VOID Pass 1 only.
8. Preview and download the result.

In [ ]:
import os
import sys
import json
import glob
import shutil
import zipfile
import subprocess
import threading
import time
from datetime import datetime
from pathlib import Path

def log(message):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)

def run(cmd, cwd=None, check=True):
    if isinstance(cmd, str):
        shown = cmd
        log(f"$ {shown}")
        result = subprocess.run(cmd, cwd=cwd, shell=True, text=True)
    else:
        shown = " ".join(map(str, cmd))
        log(f"$ {shown}")
        result = subprocess.run(cmd, cwd=cwd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {shown}")
    return result.returncode

def print_gpu_snapshot(label="GPU"):
    query = [
        "nvidia-smi",
        "--query-gpu=timestamp,name,utilization.gpu,memory.used,memory.total,power.draw",
        "--format=csv,noheader,nounits",
    ]
    try:
        result = subprocess.run(query, text=True, capture_output=True, check=False)
        if result.returncode == 0 and result.stdout.strip():
            for line in result.stdout.strip().splitlines():
                log(f"{label}: {line}")
            return
    except Exception as exc:
        log(f"{label}: compact nvidia-smi query failed: {exc}")
    run(["nvidia-smi"], check=False)

def print_system_snapshot(label):
    log(f"System snapshot: {label}")
    usage = shutil.disk_usage("/content")
    log(f"/content disk: used {usage.used / 1024**3:.1f} GB / total {usage.total / 1024**3:.1f} GB")
    ram = subprocess.run(["bash", "-lc", "free -h | sed -n '2p'"], text=True, capture_output=True, check=False)
    if ram.stdout.strip():
        log(f"RAM: {ram.stdout.strip()}")
    print_gpu_snapshot(label)

def run_stream(cmd, cwd=None, check=True, monitor_gpu=False, monitor_interval=30):
    shown = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
    log(f"$ {shown}")
    stop_monitor = threading.Event()

    def monitor():
        while not stop_monitor.wait(monitor_interval):
            print_gpu_snapshot("GPU monitor")

    monitor_thread = None
    if monitor_gpu:
        print_gpu_snapshot("GPU before launch")
        monitor_thread = threading.Thread(target=monitor, daemon=True)
        monitor_thread.start()

    start = time.time()
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        shell=isinstance(cmd, str),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {line}", end="", flush=True)
        returncode = process.wait()
    finally:
        stop_monitor.set()
        if monitor_thread:
            monitor_thread.join(timeout=2)
    elapsed = time.time() - start
    log(f"Command finished in {elapsed / 60:.1f} min with exit code {returncode}")
    if returncode == -9:
        log("Process was killed by Colab/OS with SIGKILL. This usually means CPU RAM, GPU VRAM, or total runtime pressure, not a normal Python exception.")
    if check and returncode != 0:
        raise RuntimeError(f"Command failed with exit code {returncode}: {shown}")
    return returncode

print("Python:", sys.version)
print("Working directory:", os.getcwd())
run(["nvidia-smi"], check=False)

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"VRAM: {total_gb:.2f} GB")
except Exception as exc:
    print("Torch check failed:", exc)

## Configuration

Use `RESOURCE_PROFILE = "l4_pro_balanced"` for the current Colab Pro L4 setup. It runs 45-frame chunks at `256x448`. For longer but lower-resolution L4 chunks, use `"l4_pro_long_lowres"`. Use T4 profiles only if Colab assigns a T4. Avoid frame counts like 17 or 49 here: this VOID/CogVideoX path needs the latent temporal frame count to be even.

When running through VS Code's Colab connection, the Colab file-picker can fail or stay disabled. The reliable options are:

1. Put `void_phase5_input.zip` in Google Drive at `MyDrive/void_phase5_input.zip`.
2. Upload the zip to `/content/void_phase5_input.zip` using Colab's file sidebar.
3. Set `PACKAGE_URL` to a direct downloadable URL.

A local Mac path such as `/Users/.../void_phase5_input.zip` is not visible to the remote Colab kernel.

In [ ]:
VOID_REPO = Path("/content/void-model")
DATA_ROOT = Path("/content/void_phase5_data")
OUTPUT_DIR = Path("/content/void_phase5_outputs")
UPLOAD_DIR = Path("/content/void_phase5_upload")

SEQ_NAME = "phase5_custom"
OFFICIAL_SAMPLE_NAME = "lime"

# VS Code + Colab friendly package inputs.
# Preferred: upload/copy the zip to Google Drive as MyDrive/void_phase5_input.zip.
CUSTOM_ZIP_PATH = Path("/content/void_phase5_input.zip")
GOOGLE_DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/void_phase5_input.zip")
CHUNK_ZIP_DIR = Path("/content/void_phase5_chunks")
GOOGLE_DRIVE_CHUNK_ZIP_DIR = Path("/content/drive/MyDrive/void_phase5_chunks")
CHUNK_OUTPUTS_DIR = Path("/content/void_phase5_chunk_outputs")
MERGED_OUTPUT_PATH = Path("/content/void_phase5_merged.mp4")
PROCESS_ALL_CHUNKS = True
CHUNK_ZIP_GLOB = "*.zip"
PACKAGE_URL = ""  # Optional direct URL to a zip. Leave empty if unused.
MOUNT_GOOGLE_DRIVE = True
SEARCH_COMMON_ZIP_LOCATIONS = True

# Browser upload widget is unreliable from VS Code. Turn it on only when running
# directly in the Colab browser tab.
USE_BROWSER_UPLOAD_WIDGET = False

# Keep this False for real free-Colab runs so a missing zip does not burn GPU
# time on the official sample. Set True only when you intentionally want a
# quick environment smoke test.
ALLOW_OFFICIAL_SAMPLE_FALLBACK = False

# Optional local package generation from an uploaded original video.
# Set PROJECT_REPO_URL to the GitHub repo where this code is pushed, upload the
# original video to ORIGINAL_INPUT_VIDEO_PATH, then rerun from this config cell.
AUTO_PREPARE_VOID_PACKAGE_FROM_ORIGINAL_VIDEO = False
PROJECT_REPO_URL = ""  # Optional: set to your public repo URL when auto-preparing packages.
PROJECT_REPO_BRANCH = "main"
PROJECT_REPO_DIR = Path("/content/VideoObjectRemoval")
ORIGINAL_INPUT_VIDEO_PATH = Path("/content/input_video.mp4")

# Mask/package generation settings used before VOID Pass 1. Keep CHUNK_SIZE tied
# to MAX_VIDEO_LENGTH below so Colab receives the same temporal window it will run.
PREPARED_SEQUENCE_PREFIX = "motion_void"
PREPARED_PROMPT = "clean natural background after the selected object is removed"
PREPARED_REMOVAL_MODE = "ai_object"  # ai_object | ai_text_or_logo | static_rectangle
PREPARED_TRACKING_BACKEND = "samurai"  # auto | samurai | global | dam4sam | sam2long_research
PREPARED_REFERENCE_FRAME = 5
PREPARED_MASK_PADDING = 2
PREPARED_START_FRAME = 0
PREPARED_MAX_CHUNKS = None  # Set to 1 for a quick smoke test, or None for all chunks.
PREPARED_FORCE_MASK_CACHE_REBUILD = True
PREPARED_VOID_SHADOW_DILATION_PX = 0
PREPARED_ROI_X = 278
PREPARED_ROI_Y = 104
PREPARED_ROI_WIDTH = 77
PREPARED_ROI_HEIGHT = 215
AUTO_CLONE_TRACKING_REPOS = True


# L4-first settings for Colab Pro. Use T4 profiles only if Colab assigns T4.
RESOURCE_PROFILE = "l4_pro_balanced"  # l4_pro_balanced | l4_pro_long_lowres | t4_highram_safe | t4_highram_long_lowres | t4_highram_qfloat8 | a100_quality
GPU_MONITOR_INTERVAL_SECONDS = 30
PYTORCH_CUDA_ALLOC_CONF = "expandable_segments:True"

def validate_temporal_window_size(frame_count):
    latent_frames = (frame_count - 1) // 4 + 1
    if latent_frames % 2 != 0:
        raise ValueError(
            f"Invalid temporal window {frame_count}: latent frame count is {latent_frames}, "
            "but CogVideoX patch embedding expects an even latent count. "
            "Use values like 13, 21, 29, 37, 45, 53, 61, 69, 77, or 85."
        )
    return latent_frames

if RESOURCE_PROFILE in {"t4_highram_safe", "free_colab_emergency"}:
    SAMPLE_SIZE = "192x320"
    MAX_VIDEO_LENGTH = 45
    TEMPORAL_WINDOW_SIZE = 45
    GPU_MEMORY_MODE = "sequential_cpu_offload"
    NUM_INFERENCE_STEPS = 8
    TEMPORAL_MULTIDIFFUSION_STRIDE = 16
elif RESOURCE_PROFILE in {"t4_highram_qfloat8", "free_colab_smoke"}:
    SAMPLE_SIZE = "192x320"
    MAX_VIDEO_LENGTH = 45
    TEMPORAL_WINDOW_SIZE = 45
    GPU_MEMORY_MODE = "model_cpu_offload_and_qfloat8"
    NUM_INFERENCE_STEPS = 8
    TEMPORAL_MULTIDIFFUSION_STRIDE = 16
elif RESOURCE_PROFILE == "t4_highram_long_lowres":
    SAMPLE_SIZE = "192x320"
    MAX_VIDEO_LENGTH = 85
    TEMPORAL_WINDOW_SIZE = 85
    GPU_MEMORY_MODE = "sequential_cpu_offload"
    NUM_INFERENCE_STEPS = 8
    TEMPORAL_MULTIDIFFUSION_STRIDE = 16
elif RESOURCE_PROFILE in {"l4_pro_balanced", "free_colab_balanced"}:
    SAMPLE_SIZE = "256x448"
    MAX_VIDEO_LENGTH = 45
    TEMPORAL_WINDOW_SIZE = 45
    GPU_MEMORY_MODE = "model_cpu_offload_and_qfloat8"
    NUM_INFERENCE_STEPS = 20
    TEMPORAL_MULTIDIFFUSION_STRIDE = 16
elif RESOURCE_PROFILE == "l4_pro_long_lowres":
    SAMPLE_SIZE = "192x320"
    MAX_VIDEO_LENGTH = 85
    TEMPORAL_WINDOW_SIZE = 85
    GPU_MEMORY_MODE = "model_cpu_offload_and_qfloat8"
    NUM_INFERENCE_STEPS = 12
    TEMPORAL_MULTIDIFFUSION_STRIDE = 16
elif RESOURCE_PROFILE == "a100_quality":
    SAMPLE_SIZE = "384x672"
    MAX_VIDEO_LENGTH = 85
    TEMPORAL_WINDOW_SIZE = 85
    GPU_MEMORY_MODE = "model_cpu_offload_and_qfloat8"
    NUM_INFERENCE_STEPS = 30
    TEMPORAL_MULTIDIFFUSION_STRIDE = 16
else:
    raise ValueError(f"Unknown RESOURCE_PROFILE: {RESOURCE_PROFILE}")

LATENT_TEMPORAL_FRAMES = validate_temporal_window_size(TEMPORAL_WINDOW_SIZE)

FPS = 12
GUIDANCE_SCALE = 1.0
SEED = 42
CUDA_VISIBLE_DEVICES = "0"
FORCE_CUDA_DEVICE = "cuda"
EXPECTED_OUTPUT_SECONDS = MAX_VIDEO_LENGTH / FPS

print(json.dumps({
    "custom_zip_path": str(CUSTOM_ZIP_PATH),
    "google_drive_zip_path": str(GOOGLE_DRIVE_ZIP_PATH),
    "chunk_zip_dir": str(CHUNK_ZIP_DIR),
    "google_drive_chunk_zip_dir": str(GOOGLE_DRIVE_CHUNK_ZIP_DIR),
    "process_all_chunks": PROCESS_ALL_CHUNKS,
    "chunk_zip_glob": CHUNK_ZIP_GLOB,
    "chunk_outputs_dir": str(CHUNK_OUTPUTS_DIR),
    "merged_output_path": str(MERGED_OUTPUT_PATH),
    "package_url_set": bool(PACKAGE_URL),
    "resource_profile": RESOURCE_PROFILE,
    "sample_size": SAMPLE_SIZE,
    "max_video_length": MAX_VIDEO_LENGTH,
    "temporal_window_size": TEMPORAL_WINDOW_SIZE,
    "latent_temporal_frames": LATENT_TEMPORAL_FRAMES,
    "gpu_memory_mode": GPU_MEMORY_MODE,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "temporal_multidiffusion_stride": TEMPORAL_MULTIDIFFUSION_STRIDE,
    "fallback_expected_output_seconds_at_12fps": round(EXPECTED_OUTPUT_SECONDS, 2),
    "gpu_monitor_interval_seconds": GPU_MONITOR_INTERVAL_SECONDS,
    "pytorch_cuda_alloc_conf": PYTORCH_CUDA_ALLOC_CONF,
    "cuda_visible_devices": CUDA_VISIBLE_DEVICES,
    "force_cuda_device": FORCE_CUDA_DEVICE,
    "auto_prepare_void_package": AUTO_PREPARE_VOID_PACKAGE_FROM_ORIGINAL_VIDEO,
    "project_repo_url_set": bool(PROJECT_REPO_URL),
    "project_repo_dir": str(PROJECT_REPO_DIR),
    "original_input_video_path": str(ORIGINAL_INPUT_VIDEO_PATH),
    "prepared_tracking_backend": PREPARED_TRACKING_BACKEND,
    "prepared_reference_frame": PREPARED_REFERENCE_FRAME,
    "prepared_roi": [PREPARED_ROI_X, PREPARED_ROI_Y, PREPARED_ROI_WIDTH, PREPARED_ROI_HEIGHT],
}, indent=2))

## Clone VOID And Install Dependencies

In [ ]:
if not VOID_REPO.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/Netflix/void-model.git", str(VOID_REPO)])
else:
    print(f"Repo already exists: {VOID_REPO}")

run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], cwd=VOID_REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=VOID_REPO)
run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "accelerate", "safetensors"], cwd=VOID_REPO)

# Patch the official inference script inside the Colab runtime so the config
# cell can lower steps and temporal stride for T4-sized runs.
predict_script = VOID_REPO / "inference/cogvideox_fun/predict_v2v.py"
text = predict_script.read_text()
old = "num_inference_steps = 30,"
new = "num_inference_steps = config.video_model.num_inference_steps,"
if old in text:
    text = text.replace(old, new)
    print("Patched predict_v2v.py to honor config.video_model.num_inference_steps")
else:
    print("Step-count patch was already applied or upstream code changed.")
old = "            stack_mask = config.video_model.stack_mask,\n        ).videos"
new = "            stack_mask = config.video_model.stack_mask,\n            temporal_multidiffusion_stride = config.video_model.temproal_multidiffusion_stride,\n        ).videos"
if old in text:
    text = text.replace(old, new)
    print("Patched predict_v2v.py to honor config.video_model.temproal_multidiffusion_stride")
elif "temporal_multidiffusion_stride = config.video_model.temproal_multidiffusion_stride" in text:
    print("Temporal stride patch was already applied.")
else:
    print("Temporal stride patch was not applied; upstream code changed. Check predict_v2v.py before running.")

predict_script.write_text(text)

## Download Models

This is the slowest setup step. The base CogVideoX model is large. If Colab disconnects, rerun from this cell and Hugging Face cache may save some time.

In [ ]:
base_model_dir = VOID_REPO / "CogVideoX-Fun-V1.5-5b-InP"
pass1_ckpt = VOID_REPO / "void_pass1.safetensors"

if not base_model_dir.exists():
    run(["hf", "download", "alibaba-pai/CogVideoX-Fun-V1.5-5b-InP", "--local-dir", str(base_model_dir)], cwd=VOID_REPO)
else:
    print(f"Base model already exists: {base_model_dir}")

if not pass1_ckpt.exists():
    run(["hf", "download", "netflix/void-model", "void_pass1.safetensors", "--local-dir", str(VOID_REPO)], cwd=VOID_REPO)
else:
    print(f"VOID Pass 1 checkpoint already exists: {pass1_ckpt}")

## Load The Phase 5 Test Package

Preferred package structure:

```text
void_phase5_input.zip
  input_video.mp4    # or input.mp4
  quadmask_0.mp4
  prompt.json        # {"bg": "clean background after object removal"}
```

For full-video chunked runs, put every chunk zip in one folder and keep their filenames sorted, for example `bottle_detection_phase5_chunk_000.zip`, `bottle_detection_phase5_chunk_001.zip`, and so on. The notebook will process them in that order and stitch the outputs.

Preferred chunk folders:

- `/content/void_phase5_chunks`
- `/content/drive/MyDrive/void_phase5_chunks`

For a single-package test, put one zip in one of these places:

- `/content/void_phase5_input.zip`
- `/content/drive/MyDrive/void_phase5_input.zip`
- any top-level `.zip` in `/content` or `/content/drive/MyDrive`
- a direct URL through `PACKAGE_URL`

By default, the notebook stops if no package is found so free GPU time is not wasted on the official sample. Set `ALLOW_OFFICIAL_SAMPLE_FALLBACK = True` only when you intentionally want a smoke test on the official `lime` sample.

### Auto-generate chunk zips from an uploaded original video

Set `PROJECT_REPO_URL` to this project's GitHub URL and upload the original video to `ORIGINAL_INPUT_VIDEO_PATH` from the configuration cell. When `AUTO_PREPARE_VOID_PACKAGE_FROM_ORIGINAL_VIDEO` is enabled, this notebook clones the project repo, runs the existing local VOID chunk exporter, writes chunk zips to `/content/void_phase5_chunks`, and then continues with the normal VOID Pass 1 flow.


In [ ]:
def maybe_mount_drive():
    if not MOUNT_GOOGLE_DRIVE:
        return
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        print("Google Drive already mounted:", drive_root)
        return
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Google Drive mount skipped or failed:", exc)

def download_package_from_url():
    if not PACKAGE_URL:
        return None
    target = CUSTOM_ZIP_PATH
    print("Downloading package from PACKAGE_URL to", target)
    run(["python", "-c", (
        "import urllib.request; "
        f"urllib.request.urlretrieve({PACKAGE_URL!r}, {str(target)!r})"
    )])
    return target if target.exists() else None

def find_first(root, names):
    root = Path(root)
    for name in names:
        matches = list(root.rglob(name))
        if matches:
            return matches[0]
    return None

def find_zip_candidate():
    explicit = [CUSTOM_ZIP_PATH, GOOGLE_DRIVE_ZIP_PATH]
    for path in explicit:
        if path and Path(path).exists():
            return Path(path)

    if not SEARCH_COMMON_ZIP_LOCATIONS:
        return None

    search_dirs = [Path("/content")]
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        search_dirs.extend([drive_root, drive_root / "Colab Notebooks"])

    candidates = []
    for directory in search_dirs:
        if directory.exists():
            candidates.extend(directory.glob("*.zip"))

    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None

def find_chunk_zips():
    if not PROCESS_ALL_CHUNKS:
        return []
    search_dirs = [CHUNK_ZIP_DIR, GOOGLE_DRIVE_CHUNK_ZIP_DIR]
    found = []
    for directory in search_dirs:
        directory = Path(directory)
        if directory.exists():
            found.extend(sorted(directory.glob(CHUNK_ZIP_GLOB)))
    return sorted({path.resolve() for path in found})


def clone_or_update_project_repo():
    if not AUTO_PREPARE_VOID_PACKAGE_FROM_ORIGINAL_VIDEO:
        return None
    if not ORIGINAL_INPUT_VIDEO_PATH.exists():
        print("Original input video not found; skipping local VOID package generation:", ORIGINAL_INPUT_VIDEO_PATH)
        return None
    if not PROJECT_REPO_URL:
        raise ValueError(
            "PROJECT_REPO_URL must point to your GitHub repo before generating VOID packages "
            "from ORIGINAL_INPUT_VIDEO_PATH."
        )

    if PROJECT_REPO_DIR.exists() and (PROJECT_REPO_DIR / ".git").exists():
        print("Project repo already exists:", PROJECT_REPO_DIR)
        run(["git", "fetch", "--depth", "1", "origin", PROJECT_REPO_BRANCH], cwd=PROJECT_REPO_DIR)
        run(["git", "checkout", PROJECT_REPO_BRANCH], cwd=PROJECT_REPO_DIR)
        run(["git", "pull", "--ff-only", "origin", PROJECT_REPO_BRANCH], cwd=PROJECT_REPO_DIR)
    else:
        if PROJECT_REPO_DIR.exists():
            shutil.rmtree(PROJECT_REPO_DIR)
        clone_cmd = ["git", "clone", "--depth", "1"]
        if PROJECT_REPO_BRANCH:
            clone_cmd.extend(["--branch", PROJECT_REPO_BRANCH])
        clone_cmd.extend([PROJECT_REPO_URL, str(PROJECT_REPO_DIR)])
        run(clone_cmd)
    return PROJECT_REPO_DIR

def maybe_clone_tracking_repo(name, url):
    target = PROJECT_REPO_DIR / "models" / name
    if target.exists() or not AUTO_CLONE_TRACKING_REPOS:
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--depth", "1", url, str(target)])
    return target

def install_project_package_dependencies():
    requirements = PROJECT_REPO_DIR / "requirements.txt"
    if requirements.exists():
        run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
    run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_REPO_DIR)], check=False)


def ensure_tracking_backend_code():
    if PREPARED_TRACKING_BACKEND in {"auto", "samurai"}:
        maybe_clone_tracking_repo("samurai_repo", "https://github.com/yangchris11/samurai.git")
    if PREPARED_TRACKING_BACKEND == "dam4sam":
        maybe_clone_tracking_repo("dam4sam_repo", "https://github.com/jovanavidenovic/DAM4SAM.git")
    if PREPARED_TRACKING_BACKEND == "sam2long_research":
        maybe_clone_tracking_repo("sam2long_repo", "https://github.com/Mark12Ding/SAM2Long.git")


def ensure_sam2_checkpoint():
    checkpoint = PROJECT_REPO_DIR / "models" / "sam2_repo" / "checkpoints" / "sam2.1_hiera_tiny.pt"
    if checkpoint.exists():
        os.environ.setdefault("SAM2_CHECKPOINT_PATH", str(checkpoint))
        return checkpoint
    download_script = PROJECT_REPO_DIR / "models" / "sam2_repo" / "checkpoints" / "download_ckpts.sh"
    if download_script.exists():
        run(["bash", "download_ckpts.sh"], cwd=download_script.parent)
        if checkpoint.exists():
            os.environ.setdefault("SAM2_CHECKPOINT_PATH", str(checkpoint))
            return checkpoint
    print("SAM2 checkpoint was not found in the project repo. Set SAM2_CHECKPOINT_PATH if the exporter needs AI masks.")
    return None


def generate_void_packages_from_original_video():
    repo_dir = clone_or_update_project_repo()
    if repo_dir is None:
        return []

    install_project_package_dependencies()
    ensure_tracking_backend_code()
    ensure_sam2_checkpoint()

    os.environ["PYTHONPATH"] = f"{repo_dir / 'src'}:{repo_dir / 'models' / 'sam2_repo'}:" + os.environ.get("PYTHONPATH", "")
    os.environ["SAM2_TRACKING_BACKEND"] = PREPARED_TRACKING_BACKEND
    os.environ["SAM2_BIDIRECTIONAL"] = "1"
    os.environ["SAM2_MASK_CACHE_DIR"] = str(CHUNK_ZIP_DIR / "mask_cache")
    os.environ.setdefault("SAM2_IMAGE_SIZE", "512")
    if PREPARED_TRACKING_BACKEND == "samurai":
        os.environ.setdefault("SAMURAI_REPO_DIR", str(repo_dir / "models" / "samurai_repo"))
    if PREPARED_TRACKING_BACKEND == "dam4sam":
        os.environ.setdefault("DAM4SAM_REPO_DIR", str(repo_dir / "models" / "dam4sam_repo"))
    if PREPARED_TRACKING_BACKEND == "sam2long_research":
        os.environ.setdefault("SAM2LONG_REPO_DIR", str(repo_dir / "models" / "sam2long_repo"))

    if CHUNK_ZIP_DIR.exists():
        shutil.rmtree(CHUNK_ZIP_DIR)
    CHUNK_ZIP_DIR.mkdir(parents=True, exist_ok=True)

    exporter = repo_dir / "scripts" / "export_bottle_detection_void_chunks.py"
    if not exporter.exists():
        raise FileNotFoundError(f"Could not find VOID chunk exporter in cloned repo: {exporter}")

    cmd = [
        sys.executable,
        str(exporter),
        "--input", str(ORIGINAL_INPUT_VIDEO_PATH),
        "--output-dir", str(CHUNK_ZIP_DIR),
        "--sequence-prefix", PREPARED_SEQUENCE_PREFIX,
        "--prompt", PREPARED_PROMPT,
        "--chunk-size", str(MAX_VIDEO_LENGTH),
        "--start-frame", str(PREPARED_START_FRAME),
        "--removal-mode", PREPARED_REMOVAL_MODE,
        "--reference-frame", str(PREPARED_REFERENCE_FRAME),
        "--mask-padding", str(PREPARED_MASK_PADDING),
        "--void-shadow-dilation-px", str(PREPARED_VOID_SHADOW_DILATION_PX),
        "--sam2-tracking-backend", PREPARED_TRACKING_BACKEND,
        "--mask-cache-dir", str(CHUNK_ZIP_DIR / "mask_cache"),
        "--x", str(PREPARED_ROI_X),
        "--y", str(PREPARED_ROI_Y),
        "--width", str(PREPARED_ROI_WIDTH),
        "--height", str(PREPARED_ROI_HEIGHT),
        "--overwrite",
        "--keep-package-dir",
    ]
    if PREPARED_MAX_CHUNKS is not None:
        cmd.extend(["--max-chunks", str(PREPARED_MAX_CHUNKS)])
    if PREPARED_FORCE_MASK_CACHE_REBUILD:
        cmd.append("--force-mask-cache-rebuild")

    print("Generating VOID package chunk zips from original video:", ORIGINAL_INPUT_VIDEO_PATH)
    run_stream(cmd, cwd=repo_dir, monitor_gpu=True, monitor_interval=GPU_MONITOR_INTERVAL_SECONDS)
    generated = sorted(CHUNK_ZIP_DIR.glob(CHUNK_ZIP_GLOB))
    print(f"Generated {len(generated)} chunk zip(s) in {CHUNK_ZIP_DIR}")
    return generated

def normalize_prompt(src, dst):
    if src and Path(src).exists():
        data = json.loads(Path(src).read_text())
        if isinstance(data, str):
            data = {"bg": data}
        elif "bg" not in data:
            data = {"bg": data.get("prompt", "clean natural background after the selected object is removed")}
    else:
        data = {"bg": "clean natural background after the selected object is removed"}
    Path(dst).write_text(json.dumps(data, indent=2))

def prepare_custom_zip(zip_path):
    zip_path = Path(zip_path)
    if UPLOAD_DIR.exists():
        shutil.rmtree(UPLOAD_DIR)
    if DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(UPLOAD_DIR)

    src_video = find_first(UPLOAD_DIR, ["input_video.mp4", "input.mp4"])
    src_mask = find_first(UPLOAD_DIR, ["quadmask_0.mp4", "trimask_quadmask.mp4"])
    src_prompt = find_first(UPLOAD_DIR, ["prompt.json"])
    src_manifest = find_first(UPLOAD_DIR, ["manifest.json"])
    if not src_video or not src_mask:
        raise FileNotFoundError("Zip must contain input_video.mp4 or input.mp4, plus quadmask_0.mp4")

    seq_dir = DATA_ROOT / SEQ_NAME
    seq_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_video, seq_dir / "input_video.mp4")
    shutil.copy2(src_mask, seq_dir / "quadmask_0.mp4")
    normalize_prompt(src_prompt, seq_dir / "prompt.json")
    if src_manifest:
        shutil.copy2(src_manifest, seq_dir / "manifest.json")
    return DATA_ROOT, SEQ_NAME

maybe_mount_drive()
download_package_from_url()
generate_void_packages_from_original_video()
CHUNK_ZIPS = find_chunk_zips()
zip_candidate = find_zip_candidate()
if CHUNK_ZIPS:
    print(f"Found {len(CHUNK_ZIPS)} chunk zip(s):")
    for index, path in enumerate(CHUNK_ZIPS):
        print(f"  {index:03d}: {path}")
    zip_candidate = CHUNK_ZIPS[0]

if not zip_candidate and USE_BROWSER_UPLOAD_WIDGET:
    try:
        from google.colab import files
        print("Upload void_phase5_input.zip now. This is usually reliable only in the Colab browser, not VS Code.")
        uploaded = files.upload()
        if uploaded:
            first_name = next(iter(uploaded.keys()))
            zip_candidate = Path("/content") / first_name
            print("Uploaded:", zip_candidate)
    except Exception as exc:
        print("Browser upload helper is unavailable:", exc)

if zip_candidate and Path(zip_candidate).exists():
    print("Using package zip:", zip_candidate)
    DATA_ROOT, SEQ_NAME = prepare_custom_zip(zip_candidate)
    print("Using custom package for validation:", DATA_ROOT / SEQ_NAME)
elif ALLOW_OFFICIAL_SAMPLE_FALLBACK:
    DATA_ROOT = VOID_REPO / "sample"
    SEQ_NAME = OFFICIAL_SAMPLE_NAME
    print("No custom package found. Using official sample:", DATA_ROOT / SEQ_NAME)
else:
    raise FileNotFoundError(
        "No custom package found. Put chunk zips in /content/void_phase5_chunks or "
        "/content/drive/MyDrive/void_phase5_chunks, or put one zip at "
        "/content/void_phase5_input.zip or /content/drive/MyDrive/void_phase5_input.zip."
    )

## Validate Input, Mask, And Prompt

In [ ]:
from IPython.display import Video, display
import cv2
import numpy as np

seq_dir = Path(DATA_ROOT) / SEQ_NAME
input_video_path = seq_dir / "input_video.mp4"
quadmask_path = seq_dir / "quadmask_0.mp4"
if not quadmask_path.exists():
    alt = seq_dir / "trimask_quadmask.mp4"
    if alt.exists():
        quadmask_path = alt

prompt_path = seq_dir / "prompt.json"
manifest_path = seq_dir / "manifest.json"
print("Sequence:", SEQ_NAME)
print("Input:", input_video_path)
print("Mask:", quadmask_path)
print("Prompt:", prompt_path.read_text() if prompt_path.exists() else "missing")
print("Manifest:", manifest_path.read_text() if manifest_path.exists() else "missing")

cap = cv2.VideoCapture(str(input_video_path))
print("Input frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("Input fps:", cap.get(cv2.CAP_PROP_FPS))
print("Input size:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "x", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
cap.release()

mask_cap = cv2.VideoCapture(str(quadmask_path))
ok, frame = mask_cap.read()
mask_cap.release()
if not ok:
    raise RuntimeError("Could not read first quadmask frame")
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
vals = np.unique(gray)
print("First mask frame unique values, first 20:", vals[:20])
print("Expected values are near 0, 63, 127, 255. For simple no-VLM tests, 0 and 255 are enough.")

display(Video(str(input_video_path), embed=True, width=672))

## Auto-Chunk Single Long Video
If you provided a single long video but no chunks, this cell will automatically split your package into processable segments based on `MAX_VIDEO_LENGTH`.

In [ ]:
import math
import zipfile
import cv2

print("--- Auto-Chunking Single Input ---")
seq_dir = Path(DATA_ROOT) / SEQ_NAME
input_video = seq_dir / "input_video.mp4"
quadmask = seq_dir / "quadmask_0.mp4"
prompt_file = seq_dir / "prompt.json"
manifest_file = seq_dir / "manifest.json"

# Get total frames
cap = cv2.VideoCapture(str(input_video))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

if not CHUNK_ZIPS and total_frames > MAX_VIDEO_LENGTH:
    print(f"Video has {total_frames} frames. Auto-chunking into {MAX_VIDEO_LENGTH}-frame pieces...")
    CHUNK_ZIP_DIR.mkdir(parents=True, exist_ok=True)

    num_chunks = math.ceil(total_frames / MAX_VIDEO_LENGTH)
    new_chunk_zips = []

    for i in range(num_chunks):
        start_frame = i * MAX_VIDEO_LENGTH
        end_frame = min((i + 1) * MAX_VIDEO_LENGTH, total_frames) - 1
        chunk_frames = end_frame - start_frame + 1

        chunk_dir = CHUNK_ZIP_DIR / f"chunk_tmp_{i:03d}"
        chunk_dir.mkdir(parents=True, exist_ok=True)

        chunk_video = chunk_dir / "input_video.mp4"
        chunk_mask = chunk_dir / "quadmask_0.mp4"

        # Slice video
        run([
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-i", str(input_video),
            "-vf", f"select='between(n\\,{start_frame}\\,{end_frame})',setpts=PTS-STARTPTS",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p",
            str(chunk_video)
        ])

        # Slice mask
        run([
            "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
            "-i", str(quadmask),
            "-vf", f"select='between(n\\,{start_frame}\\,{end_frame})',setpts=PTS-STARTPTS",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p",
            str(chunk_mask)
        ])

        # Copy prompt
        shutil.copy2(prompt_file, chunk_dir / "prompt.json")

        # Update manifest for chunk
        if manifest_file.exists():
            manifest_data = json.loads(manifest_file.read_text())
            manifest_data["frame_count"] = chunk_frames
            (chunk_dir / "manifest.json").write_text(json.dumps(manifest_data, indent=2))

        # Zip it
        zip_path = CHUNK_ZIP_DIR / f"{SEQ_NAME}_chunk_{i:03d}.zip"
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for file_path in chunk_dir.glob("*"):
                zf.write(file_path, file_path.name)

        new_chunk_zips.append(zip_path)
        shutil.rmtree(chunk_dir)
        print(f"Created {zip_path.name} ({chunk_frames} frames)")

    CHUNK_ZIPS = new_chunk_zips
    print(f"Successfully generated {len(CHUNK_ZIPS)} chunk zips!")
else:
    print("No auto-chunking needed. Video fits in MAX_VIDEO_LENGTH or CHUNK_ZIPS already provided.")


## Run VOID Pass 1

This is the main Phase 5 run. It uses prebuilt masks and therefore does not call Gemini, Ollama, SAM2, SAM3, or the VLM mask-reasoner.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Make CUDA selection explicit for the subprocess launched from VS Code/Colab.
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = PYTORCH_CUDA_ALLOC_CONF

import gc
import torch
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("PYTORCH_CUDA_ALLOC_CONF:", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available in this notebook kernel. In Colab, switch Runtime -> "
        "Change runtime type -> GPU, then rerun from the first cell."
    )

print("GPU:", torch.cuda.get_device_name(0))
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"VRAM: {total_gb:.2f} GB")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print_system_snapshot("before VOID pass 1")

if GPU_MEMORY_MODE == "model_cpu_offload_and_qfloat8":
    print(
        "Using model_cpu_offload_and_qfloat8: this is the default balanced free-Colab profile. "
        "It should use GPU more actively than sequential offload while still reducing VRAM pressure."
    )

if GPU_MEMORY_MODE == "sequential_cpu_offload":
    print(
        "Using sequential_cpu_offload: this is the safest T4 profile after qfloat8 OOM. "
        "It keeps most weights on CPU and moves layers to GPU one at a time, "
        "so nvidia-smi may show low or spiky GPU use."
    )

def current_package_fps():
    manifest_path = Path(DATA_ROOT) / SEQ_NAME / "manifest.json"
    if manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text())
            fps = float(manifest.get("fps", 0))
            if fps > 0:
                return fps
        except Exception as exc:
            print("Could not read manifest fps, using fallback FPS:", exc)
    return FPS

def current_package_frame_count():
    manifest_path = Path(DATA_ROOT) / SEQ_NAME / "manifest.json"
    if manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text())
            frame_count = int(manifest.get("frame_count", 0))
            if frame_count > 0:
                return frame_count
        except Exception as exc:
            print("Could not read manifest frame_count, using window size:", exc)
    return MAX_VIDEO_LENGTH

def build_void_cmd(save_path):
    run_fps = round(current_package_fps()) # Round to nearest integer
    return [
        sys.executable,
        "inference/cogvideox_fun/predict_v2v.py",
        "--config", "config/quadmask_cogvideox.py",
        f"--config.data.data_rootdir={DATA_ROOT}",
        f"--config.experiment.run_seqs={SEQ_NAME}",
        f"--config.experiment.save_path={save_path}",
        "--config.experiment.skip_if_exists=False",
        f"--config.video_model.model_name={VOID_REPO / 'CogVideoX-Fun-V1.5-5b-InP'}",
        f"--config.video_model.transformer_path={VOID_REPO / 'void_pass1.safetensors'}",
        f"--config.data.sample_size={SAMPLE_SIZE}",
        f"--config.data.max_video_length={MAX_VIDEO_LENGTH}",
        f"--config.data.fps={run_fps}",
        f"--config.video_model.temporal_window_size={TEMPORAL_WINDOW_SIZE}",
        f"--config.video_model.temproal_multidiffusion_stride={TEMPORAL_MULTIDIFFUSION_STRIDE}",
        f"--config.video_model.num_inference_steps={NUM_INFERENCE_STEPS}",
        f"--config.video_model.guidance_scale={GUIDANCE_SCALE}",
        f"--config.system.gpu_memory_mode={GPU_MEMORY_MODE}",
        f"--config.system.device={FORCE_CUDA_DEVICE}",
        "--config.system.ulysses_degree=1",
        "--config.system.ring_degree=1",
        f"--config.system.seed={SEED}",
    ]

def release_cuda_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

def find_main_output(output_dir):
    output_dir = Path(output_dir)
    outputs = sorted(output_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    main_outputs = [p for p in outputs if not p.name.endswith("_tuple.mp4")]
    if not main_outputs:
        raise FileNotFoundError(f"No main MP4 output found in {output_dir}")
    return main_outputs[-1]

def run_void_for_current_package(save_path, label):
    save_path = Path(save_path)
    if save_path.exists():
        shutil.rmtree(save_path)
    save_path.mkdir(parents=True, exist_ok=True)
    run_fps = current_package_fps()
    cmd = build_void_cmd(save_path)
    print(json.dumps({
        "label": label,
        "void_repo": str(VOID_REPO),
        "data_root": str(DATA_ROOT),
        "sequence": SEQ_NAME,
        "output_dir": str(save_path),
        "resource_profile": RESOURCE_PROFILE,
        "sample_size": SAMPLE_SIZE,
        "max_video_length": MAX_VIDEO_LENGTH,
        "temporal_window_size": TEMPORAL_WINDOW_SIZE,
        "latent_temporal_frames": LATENT_TEMPORAL_FRAMES,
        "temporal_multidiffusion_stride": TEMPORAL_MULTIDIFFUSION_STRIDE,
        "gpu_memory_mode": GPU_MEMORY_MODE,
        "num_inference_steps": NUM_INFERENCE_STEPS,
        "fps": run_fps,
        "expected_output_seconds": round(MAX_VIDEO_LENGTH / run_fps, 2),
    }, indent=2))
    try:
        run_stream(
            cmd,
            cwd=VOID_REPO,
            monitor_gpu=True,
            monitor_interval=GPU_MONITOR_INTERVAL_SECONDS,
        )
    except RuntimeError:
        print_system_snapshot(f"after failed VOID pass 1: {label}")
        raise
    finally:
        release_cuda_memory()
    return find_main_output(save_path)

def stitch_chunk_outputs(chunk_outputs):
    CHUNK_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    list_path = CHUNK_OUTPUTS_DIR / "concat_list.txt"
    list_path.write_text("".join(f"file {str(path.resolve())!r}\n" for path in chunk_outputs))
    if MERGED_OUTPUT_PATH.exists():
        MERGED_OUTPUT_PATH.unlink()
    run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-f", "concat", "-safe", "0", "-i", str(list_path),
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",
        "-pix_fmt", "yuv420p", str(MERGED_OUTPUT_PATH),
    ])
    return MERGED_OUTPUT_PATH

def save_stable_chunk_output(source_path, target_path):
    frame_count = current_package_frame_count()
    if frame_count >= MAX_VIDEO_LENGTH:
        shutil.copy2(source_path, target_path)
        return target_path
    print(f"Trimming final/short chunk to {frame_count} frame(s): {target_path}")
    run([
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-i", str(source_path), "-frames:v", str(frame_count),
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",
        "-pix_fmt", "yuv420p", str(target_path),
    ])
    return target_path

FINAL_OUTPUT_PATH = None
if CHUNK_ZIPS:
    print(f"Running {len(CHUNK_ZIPS)} chunk(s) in sorted order on {torch.cuda.get_device_name(0)}.")
    CHUNK_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    chunk_outputs = []
    for chunk_index, chunk_zip in enumerate(CHUNK_ZIPS):
        label = f"chunk {chunk_index + 1}/{len(CHUNK_ZIPS)}: {Path(chunk_zip).name}"
        print("=" * 80)
        print(label)
        DATA_ROOT, SEQ_NAME = prepare_custom_zip(chunk_zip)
        chunk_save_path = CHUNK_OUTPUTS_DIR / f"chunk_{chunk_index:03d}"
        chunk_output = run_void_for_current_package(chunk_save_path, label)
        stable_output = CHUNK_OUTPUTS_DIR / f"void_chunk_{chunk_index:03d}.mp4"
        save_stable_chunk_output(chunk_output, stable_output)
        chunk_outputs.append(stable_output)
        print(f"Saved chunk output: {stable_output}")
        print_system_snapshot(f"after {label}")
    FINAL_OUTPUT_PATH = stitch_chunk_outputs(chunk_outputs)
    print("Merged chunk output:", FINAL_OUTPUT_PATH)
else:
    print("Running one VOID package. To process a full video, put all chunk zips in the configured chunk folder.")
    FINAL_OUTPUT_PATH = run_void_for_current_package(OUTPUT_DIR, "single package")

print_system_snapshot("after VOID pass 1")

## Preview And Download Output

In [ ]:
if "FINAL_OUTPUT_PATH" in globals() and FINAL_OUTPUT_PATH and Path(FINAL_OUTPUT_PATH).exists():
    selected_output = Path(FINAL_OUTPUT_PATH)
    print("Selected final output:", selected_output)
    display(Video(str(selected_output), embed=True, width=672))
else:
    outputs = sorted(Path(OUTPUT_DIR).glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    print("Outputs:")
    for path in outputs:
        print("-", path)

    if not outputs:
        raise FileNotFoundError(f"No MP4 outputs found in {OUTPUT_DIR}")

    main_outputs = [p for p in outputs if not p.name.endswith("_tuple.mp4")]
    comparison_outputs = [p for p in outputs if p.name.endswith("_tuple.mp4")]
    selected_output = main_outputs[-1] if main_outputs else outputs[-1]
    print("Selected output:", selected_output)
    display(Video(str(selected_output), embed=True, width=672))

    if comparison_outputs:
        print("Comparison:", comparison_outputs[-1])
        display(Video(str(comparison_outputs[-1]), embed=True, width=1200))

In [ ]:
try:
    from google.colab import files
    archive_path = Path("/content/void_phase5_outputs.zip")
    if archive_path.exists():
        archive_path.unlink()
    archive_source = CHUNK_OUTPUTS_DIR if CHUNK_OUTPUTS_DIR.exists() else OUTPUT_DIR
    if "FINAL_OUTPUT_PATH" in globals() and FINAL_OUTPUT_PATH and Path(FINAL_OUTPUT_PATH).exists():
        print("Final merged output:", FINAL_OUTPUT_PATH)
    shutil.make_archive(str(archive_path.with_suffix("")), "zip", archive_source)
    print("Downloading:", archive_path)
    files.download(str(archive_path))
except Exception as exc:
    print("Download helper unavailable. Outputs remain here:", CHUNK_OUTPUTS_DIR if CHUNK_OUTPUTS_DIR.exists() else OUTPUT_DIR)
    print(exc)

## Retry Notes

If the run fails on Colab Pro:

- For the current L4 setup, use `RESOURCE_PROFILE = "l4_pro_balanced"` with chunked zip inputs in `/content/void_phase5_chunks` or `/content/drive/MyDrive/void_phase5_chunks`.
- One L4 balanced chunk is 45 frames at `256x448`. For exported local chunks, the notebook reads the original FPS from `manifest.json`, so the stitched video should keep the input timing.
- On T4 High RAM, start with `RESOURCE_PROFILE = "t4_highram_safe"`. High RAM helps CPU memory, but the T4 still only has about 15 GB VRAM.
- The T4-safe profile uses `SAMPLE_SIZE = "192x320"`, `MAX_VIDEO_LENGTH = 45`, `TEMPORAL_WINDOW_SIZE = 45`, and `NUM_INFERENCE_STEPS = 8`.
- Avoid `17` and `49` frame windows in this VOID path. They produce odd latent frame counts (`5` and `13`) and can fail with a CogVideoX reshape error. Use valid windows such as `45` or `85`.
- If `t4_highram_safe` still OOMs, disconnect/reconnect the Colab runtime to clear VRAM, rerun from the first cell, and keep the same 45-frame settings.
- If the T4-safe run succeeds but the output is too short, try `RESOURCE_PROFILE = "t4_highram_long_lowres"` for an 85-frame low-resolution test.
- If Colab gives an NVIDIA L4, use `RESOURCE_PROFILE = "l4_pro_balanced"` for the 45-frame `256x448` run, or `RESOURCE_PROFILE = "l4_pro_long_lowres"` for an 85-frame low-resolution run.
- After the long low-resolution run succeeds, try `RESOURCE_PROFILE = "t4_highram_qfloat8"` for speed only if Colab assigns T4; stay on the L4 profiles while you have L4.
- If model download fails, rerun only the download cell after reconnecting.
- If the output is poor, keep it as a Phase 5 observation. The first goal is proving that VOID Pass 1 can run without Gemini on a free runtime.